# Instrument model
> Model the difference between an instrument and KSSL with a few physical effects shared by all soils

In [ ]:
#| default_exp instrument_model

In [ ]:
#| export
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.interpolate import BSpline

In [ ]:
from fastcore.test import *
from scipy.signal import find_peaks
from sklearn.model_selection import GroupKFold
from soilspecdata.datasets.ringtrial import load_ringtrial
from ossl_rt_transfer.data import rt_path, load_instruments, near_dup_groups

The cells that read the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

## The model

For each soil, the KSSL spectrum is the reference. The model turns it into the spectrum that another instrument records for the same soil, with parameters shared by all soils of that instrument. The effects, applied in this order to raw absorbance:

- `shift`: a wavenumber shift and stretch, applied by linear interpolation.
- `broaden`: a Gaussian filter, for a coarser resolution.
- `contrast`: absorbance raised to a power γ.
- `gain`: multiplication by a smooth curve.
- `baseline`: addition of a smooth curve.

Gain and baseline are B-splines with 10 basis functions over the whole range.

In [ ]:
#| export
def bspline_basis(
    x, # Points to evaluate the basis on, increasing
    n_basis=10, # Number of basis functions
    k=3, # Degree
)->np.ndarray: # One row per point, one column per basis function
    "Clamped B-spline basis with knots evenly spaced over `x`"
    t = np.r_[[x[0]]*k, np.linspace(x[0], x[-1], n_basis-k+1), [x[-1]]*k]
    return BSpline.design_matrix(x, t, k).toarray()

In [ ]:
test_close(bspline_basis(np.linspace(0, 1, 50)).sum(1), np.ones(50))

In [ ]:
#| export
co2_region = (2269, 2389) # Atmospheric CO₂ band, in cm⁻¹, left out of the fit as in mirzai

def snv_t(x): # Spectra, one per row
    "SNV of each row of a tensor"
    return (x - x.mean(-1, keepdim=True)) / x.std(-1, keepdim=True)

class InstrumentModel(torch.nn.Module):
    "Turn reference spectra into the spectra of another instrument, with the effects in `effects`"
    def __init__(self,
        wn, # Wavenumbers, evenly spaced
        effects=('gain', 'baseline'), # Any of 'shift', 'broaden', 'contrast', 'gain' and 'baseline'
        n_basis=10, # Number of B-spline basis functions for gain and baseline
        max_hw=20, # Half-width of the broadening filter, in points
    ):
        super().__init__()
        self.effects, self.dw, self.max_hw = set(effects), float(wn[1]-wn[0]), max_hw
        f64 = dict(dtype=torch.float64)
        self.register_buffer('B', torch.tensor(bspline_basis(wn, n_basis), **f64))
        self.register_buffer('nu', torch.tensor((wn - wn.mean())/(wn.max()-wn.min())*2, **f64))
        self.g = torch.nn.Parameter(torch.ones(n_basis, **f64))
        self.b = torch.nn.Parameter(torch.zeros(n_basis, **f64))
        self.shift = torch.nn.Parameter(torch.zeros(2, **f64)) # Offset in points, and stretch
        self.log_sigma = torch.nn.Parameter(torch.zeros((), **f64)) # Width of the filter, in points
        self.log_gamma = torch.nn.Parameter(torch.zeros((), **f64))

    def warp(self, x):
        n = x.shape[-1]
        src = (torch.arange(n, dtype=x.dtype) + self.shift[0] + self.shift[1]*self.nu*n/2).clamp(0, n-1)
        i0 = src.floor().long().clamp(max=n-2); f = src - i0
        return x[..., i0]*(1-f) + x[..., i0+1]*f

    def broaden(self, x):
        k = torch.arange(-self.max_hw, self.max_hw+1, dtype=x.dtype)
        w = torch.exp(-0.5*(k/self.log_sigma.exp())**2)
        x = F.pad(x[:, None], (self.max_hw, self.max_hw), mode='replicate')
        return F.conv1d(x, (w/w.sum())[None, None])[:, 0]

    def forward(self, x):
        if 'shift' in self.effects: x = self.warp(x)
        if 'broaden' in self.effects: x = self.broaden(x)
        if 'contrast' in self.effects: x = x.clamp(min=1e-6)**self.log_gamma.exp()
        if 'gain' in self.effects: x = x * (self.B @ self.g)
        if 'baseline' in self.effects: x = x + self.B @ self.b
        return x

    def params(self)->dict: # The fitted physical parameters, NaN for the effects not in the model
        "Shift and stretch, filter width in cm⁻¹, and γ"
        use = lambda e, v: v if e in self.effects else float('nan')
        return {'shift_cm': use('shift', self.shift[0].item()*self.dw), 'stretch': use('shift', self.shift[1].item()),
                'sigma_cm': use('broaden', self.log_sigma.exp().item()*self.dw), 'gamma': use('contrast', self.log_gamma.exp().item())}

In [ ]:
wn = np.arange(650, 4001, 2.)
x = torch.rand(3, len(wn), dtype=torch.float64) + 0.5
m = InstrumentModel(wn, ('shift', 'broaden', 'contrast', 'gain', 'baseline'))
test_eq(m(x).shape, x.shape)
test_close(InstrumentModel(wn, ('shift', 'contrast', 'gain', 'baseline'))(x), x)

With the parameters at their starting values, every effect except the broadening leaves the spectra unchanged.

## Fitting

`fit_instrument` fits one model to the paired spectra of an instrument and KSSL. The loss compares the SNV of the modelled spectra with the SNV of the measured ones. SNV removes the offset and scale of each scan, so the model needs no parameters for single soils, and the error is on the scale the CNN sees. The CO₂ band is left out.

In [ ]:
#| export
def paired_spectra(
    rt, # Ring trial, from `load_ringtrial`
    inst, # Instrument
    ref='KSSL', # Reference instrument
)->tuple: # Sample ids, spectra of `inst`, spectra of `ref`, in the same order
    "Spectra of `inst` and `ref` for the soils that both measured"
    a, b = rt[inst], rt[ref]
    ids = np.intersect1d(a.sample_ids, b.sample_ids)
    return ids, a.spectra[pd.Index(a.sample_ids).get_indexer(ids)], b.spectra[pd.Index(b.sample_ids).get_indexer(ids)]

def _mask(wn): return torch.tensor((wn < co2_region[0]) | (wn > co2_region[1]))

def fit_instrument(
    wn, # Wavenumbers
    Xr, # Reference spectra
    Xi, # Spectra of the instrument, for the same soils
    effects, # Effects of `InstrumentModel`
    iters=300, # Iterations of L-BFGS
)->InstrumentModel:
    "Fit an `InstrumentModel` that turns `Xr` into `Xi`, on SNV spectra without the CO₂ band"
    m = InstrumentModel(wn, effects)
    if not effects: return m
    Xr, Xi, mask = torch.tensor(Xr), torch.tensor(Xi), _mask(wn)
    opt = torch.optim.LBFGS(m.parameters(), max_iter=iters, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = ((snv_t(m(Xr)) - snv_t(Xi))[:, mask]**2).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return m

def residuals(m, wn, Xr, Xi)->np.ndarray: # One row per soil, without the CO₂ band
    "SNV of the modelled spectra minus SNV of the measured ones"
    with torch.no_grad(): r = snv_t(m(torch.tensor(Xr))) - snv_t(torch.tensor(Xi))
    return r[:, _mask(wn)].numpy()

In [ ]:
xi = InstrumentModel(wn, ('gain', 'baseline'))
with torch.no_grad(): xi.g.mul_(1.3); xi.b.add_(0.2); y = xi(x)
m = fit_instrument(wn, x.numpy(), y.numpy(), ('gain', 'baseline'))
assert np.abs(residuals(m, wn, x.numpy(), y.numpy())).max() < 1e-4

A model fitted to spectra made with a known gain and baseline reproduces them.

## Instruments

MSU has the same hardware as KSSL: a Bruker Vertex 70 with the HTS-XT accessory, an MCT detector and a KBr beamsplitter, at 4 cm⁻¹, with pressed samples. What differs is mostly the laboratory. IAEA, the instrument of the Fukushima scans, is far from KSSL. Both record at 4 cm⁻¹, like KSSL, so OSU, at 8 cm⁻¹, is added to test whether the fitted broadening follows the resolution.

In [ ]:
#| rtdata
rt = load_ringtrial(rt_path())
wn = rt['KSSL'].wavenumbers.astype(float)
insts = ['MSU', 'IAEA_aug2022', 'OSU']
load_instruments().loc[['KSSL', *insts], ['manufacturer', 'model', 'accessory', 'detector', 'resolution', 'presentation', 'cluster']]

,manufacturer,model,accessory,detector,resolution,presentation,cluster
organization,,,,,,,
KSSL,Bruker,Vertex 70,Bruker HTS-XT,MCT,4.0,Pressed,4
MSU,Bruker,Vertex 70,Bruker HTS-XT,MCT,4.0,Pressed,4
IAEA_aug2022,Thermo Fisher,Nicolet iS20,Collector II Diffuse Reflectance Accessory,MCT,4.0,Pressed / levelled,2
OSU,Thermo Fisher,Nicolet iS50,"Pike X,Y Autosampler",DTGS,8.0,Levelled,2


## Shared and soil-specific differences

Each model adds effects to the one above it. `total` is the RMS of the SNV residuals. `shared` is the RMS of their mean over soils: the part a model with shared parameters can remove. `soil_specific` is what remains after the mean is removed. For comparison, the spread between soils of the KSSL SNV spectra is 0.26.

In [ ]:
#| rtdata
models = {'none': (), 'gain+baseline': ('gain', 'baseline'), '+shift': ('gain', 'baseline', 'shift'),
          '+broaden': ('gain', 'baseline', 'shift', 'broaden'), '+contrast': ('gain', 'baseline', 'shift', 'broaden', 'contrast')}
def decompose(inst):
    ids, Xi, Xr = paired_spectra(rt, inst); rows = {}
    for name, eff in models.items():
        m = fit_instrument(wn, Xr, Xi, eff); r = residuals(m, wn, Xr, Xi); sh = r.mean(0)
        rows[name] = {'total': np.sqrt((r**2).mean()), 'shared': np.sqrt((sh**2).mean()),
                      'soil_specific': np.sqrt(((r-sh)**2).mean()), **m.params()}
    return pd.DataFrame(rows).T
dec = pd.concat({i: decompose(i) for i in insts})
dec.round(3)

total  shared  ...  sigma_cm  gamma
MSU          none           0.108   0.046  ...       NaN    NaN
             gain+baseline  0.098   0.021  ...       NaN    NaN
             +shift         0.098   0.021  ...       NaN    NaN
             +broaden       0.098   0.021  ...     0.435    NaN
             +contrast      0.098   0.021  ...     0.119  1.060
IAEA_aug2022 none           0.487   0.423  ...       NaN    NaN
             gain+baseline  0.245   0.105  ...       NaN    NaN
             +shift         0.245   0.105  ...       NaN    NaN
             +broaden       0.245   0.105  ...     1.684    NaN
             +contrast      0.243   0.105  ...     1.664  0.594
OSU          none           0.254   0.207  ...       NaN    NaN
             gain+baseline  0.153   0.062  ...       NaN    NaN
             +shift         0.153   0.061  ...       NaN    NaN
             +broaden       0.153   0.061  ...     0.485    NaN
             +contrast      0.152   0.062  ...     1.820  1.348

[15 rows x 7 columns]

Gain and baseline remove most of the shared difference: from 0.42 to 0.11 on IAEA, and from 0.21 to 0.06 on OSU. They barely change the soil-specific part, which on IAEA (0.22) is close to the spread between soils. Shift, broadening and contrast add almost nothing. IAEA has a consistent shift of about -2.4 cm⁻¹. The fitted broadening of OSU does not follow its 8 cm⁻¹ resolution.

## Soils left out

The same models, fitted on 4 of 5 folds of soils and scored on the fifth. The folds keep the near-duplicate groups of `00_data` together:

In [ ]:
#| rtdata
groups = pd.Series(near_dup_groups(rt['KSSL']), index=rt['KSSL'].sample_ids)
def cv_rmse(inst, eff, n_splits=5):
    ids, Xi, Xr = paired_spectra(rt, inst); sq = 0.
    for tr, te in GroupKFold(n_splits).split(Xr, groups=groups.loc[ids].values):
        sq += (residuals(fit_instrument(wn, Xr[tr], Xi[tr], eff), wn, Xr[te], Xi[te])**2).mean()*len(te)
    return np.sqrt(sq/len(ids))
pd.DataFrame({i: {n: cv_rmse(i, e) for n, e in models.items()} for i in insts}).round(3)

,MSU,IAEA_aug2022,OSU
none,0.108,0.487,0.254
gain+baseline,0.100,0.249,0.156
+shift,0.100,0.249,0.156
+broaden,0.100,0.249,0.156
+contrast,0.100,0.248,0.155


The gain and baseline carry over to soils left out of the fit. On IAEA, the error on those soils (0.249) is close to the error when the fit uses every soil (0.245).

## The soil-specific part

A PCA of the residuals of the full model, after removing their mean. `explained` is the share of variance of each component. The table also gives the wavenumber of the largest loading, the share of the loading's energy in the 20 lowest Fourier frequencies, and the correlation of the scores with the soil properties:

In [ ]:
#| rtdata
src = rt['KSSL'].source
wm = wn[(wn < co2_region[0]) | (wn > co2_region[1])]
def residual_pca(inst, n=3):
    ids, Xi, Xr = paired_spectra(rt, inst)
    r = residuals(fit_instrument(wn, Xr, Xi, models['+contrast']), wn, Xr, Xi); r = r - r.mean(0)
    U, S, Vt = np.linalg.svd(r, full_matrices=False)
    props = src.loc[ids, ['clay_perc', 'carbon_org_perc', 'potassium_cmolkg', 'pH_H20']].astype(float)
    rows = []
    for j in range(n):
        f = np.abs(np.fft.rfft(Vt[j]))**2
        sc = pd.Series(U[:, j]*S[j], index=ids)
        rows.append({'explained': S[j]**2/(S**2).sum(), 'peak_cm': wm[np.abs(Vt[j]).argmax()], 'low_freq': f[:20].sum()/f.sum(),
                     **{p: sc.corr(props[p]) for p in props}})
    return pd.DataFrame(rows, index=[f'pc{j+1}' for j in range(n)])
pd.concat({i: residual_pca(i) for i in ['MSU', 'IAEA_aug2022']}).round(2)

explained  peak_cm  ...  potassium_cmolkg  pH_H20
MSU          pc1       0.50   3400.0  ...              0.02    0.21
             pc2       0.23   1108.0  ...              0.07   -0.10
             pc3       0.12   1342.0  ...             -0.17   -0.46
IAEA_aug2022 pc1       0.52   1016.0  ...             -0.19    0.03
             pc2       0.18   1288.0  ...             -0.40   -0.29
             pc3       0.11   1258.0  ...              0.10    0.31

[6 rows x 7 columns]

Three components explain most of the soil-specific part, so it is structured, not noise. The components are smooth, with almost all their energy at low frequencies, and correlate weakly with the soil properties. On MSU, the first lies around 3400 cm⁻¹, the O-H band of water, and the second around 1100 cm⁻¹. On IAEA, the first lies around 1020 cm⁻¹. Both regions are near the Si-O band of quartz and silicates, which in diffuse reflectance depends on particle size and packing. A possible reading is that the soil-specific part comes from how each laboratory prepared and presented each sample, such as moisture and packing, and not from the instrument. This is not tested yet.

## Naming the components

Two checks on whether the soil-specific components are preparation effects. The residuals of the full model, minus each instrument's mean, are pooled over the 20 instruments, and one PCA is fitted to them. Each lab's mean score is then 0, so the table gives each lab's spread of scores, and the correlation of its scores with clay:

In [ ]:
#| rtdata
others = [k for k in rt if k != 'KSSL']
R, lab, soil = [], [], []
for k in others:
    ids, Xi, Xr = paired_spectra(rt, k)
    r = residuals(fit_instrument(wn, Xr, Xi, models['+contrast']), wn, Xr, Xi)
    R.append(r - r.mean(0)); lab += [k]*len(ids); soil += list(ids)
U, S, Vt = np.linalg.svd(np.vstack(R), full_matrices=False)
Vt = Vt * np.sign(Vt[:, [np.abs(wm-3400).argmin()]]) # Each loading positive at 3400 cm⁻¹
print('explained:', (S[:4]**2/(S**2).sum()).round(3))
for j in range(3):
    pk, _ = find_peaks(np.abs(Vt[j]), distance=15); pk = np.sort(pk[np.argsort(-np.abs(Vt[j][pk]))][:6])
    print(f'pc{j+1} largest loadings:', [(int(wm[p]), round(float(Vt[j][p]), 3)) for p in pk])

explained: [0.359 0.198 0.105 0.081]
pc1 largest loadings: [(694, -0.043), (800, -0.048), (916, -0.041), (1044, -0.082), (1100, -0.069), (3386, 0.027)]
pc2 largest loadings: [(1392, -0.054), (1438, -0.059), (1474, -0.055), (3400, 0.045), (3446, 0.048), (3526, 0.044)]
pc3 largest loadings: [(3746, -0.051), (3818, -0.06), (3850, -0.065), (3898, -0.061), (3942, -0.054), (3986, -0.052)]


In [ ]:
#| rtdata
sc = pd.DataFrame(U[:, :3]*S[:3], columns=['pc1', 'pc2', 'pc3']).assign(lab=lab, clay=src.loc[soil, 'clay_perc'].astype(float).values)
g = sc.groupby('lab')
per_lab = pd.DataFrame({**{f'sd_{c}': g[c].std() for c in ['pc1', 'pc2', 'pc3']},
                        **{f'r_{c}_clay': g.apply(lambda d, c=c: d[c].corr(d.clay)) for c in ['pc1', 'pc2']}})
pd.concat([load_instruments().reindex(per_lab.index)[['accessory', 'presentation', 'cluster']], per_lab.round(2)], axis=1).sort_values('sd_pc1')

,accessory,presentation,cluster,sd_pc1,sd_pc2,sd_pc3,r_pc1_clay,r_pc2_clay
lab,,,,,,,,
LandCare,Bruker HTS-XT,Pressed/levelled,4.0,2.26,2.19,0.56,-0.10,0.28
MSU,Bruker HTS-XT,Pressed,4.0,2.27,2.24,0.57,-0.32,0.13
UGhent,Handheld FTIR,Pressed / levelled,2.0,2.41,2.54,0.66,-0.37,0.38
Rothamsted,HTS-XT,Levelled,4.0,2.54,2.51,0.72,-0.35,0.36
CSU_IL,Bruker HTS-XT,Pressed,4.0,2.66,2.38,0.72,0.05,0.17
Scion,Bruker HTS-XT,Pressed,4.0,2.99,2.42,1.05,-0.42,0.19
ETH_vertex,Bruker HTS-XT,Levelled,4.0,3.05,2.69,0.77,-0.41,0.35
ETH_alpha_1,Bruker Alpha QuickSnap,Levelled,3.0,3.15,2.65,0.90,-0.48,0.40
ETH_alpha_2,Bruker Alpha II QuickSnap DRIFT,Levelled,3.0,3.24,2.67,0.95,-0.44,0.42


The first component lies on the main quartz bands, at 1044-1100, 800 and 694 cm⁻¹. In diffuse reflectance these bands change with particle size. In most labs its scores correlate negatively with clay, so sandier soils, richer in quartz, differ more, as a difference in grinding or packing would make them. UWisc, the only lab with a front-facing accessory, has the opposite sign. The second component is a broad band at 3400-3526 cm⁻¹, with a small rise at 1630 cm⁻¹, as adsorbed water would give. Its scores correlate positively with clay in most labs, and clay soils hold more hygroscopic water. The spread of the first component is smallest for the labs with the HTS-XT accessory of KSSL, and largest for IAEA, UWisc and Lesotho. Lesotho also has a third component far larger than any other lab's. Presentation, pressed or levelled, makes little difference, and it is confounded with the accessory.